# Save annotated Seurat objects and AnnData for downstream analysis

In [1]:
suppressPackageStartupMessages({
  library(Seurat)
  library(future)
  library(tidyverse)
})

In [ ]:
options(future.globals.maxSize = 1000000 * 1024^2, hpc.ncpus = 16)
plan(multicore, workers = getOption("hpc.ncpus", 1))
reticulate::use_condaenv(condaenv = "ocrelizumab_paper")

# Helper functions

In [3]:
source("../scripts/python.R")

Loading required package: reticulate



# Discovery cohort

### Full dataset

QS object generated by running Snakemake workflows in `01_prepare_internal_datasets`

In [ ]:
seu <- qs::qread(file = "../data/processed/cite_seq/cohort_treatment_naive/cellbender/multi/peak-method:None_b:False_d:False/n-features:5000_normalisation:LogNormalize_clr:seurat_M:True_C:True/batch:orig.ident_normalisation:LogNormalize_integration:harmony/integrated.qs", nthreads = getOption("hpc.ncpus", 1))

### Cluster annotation

In [ ]:
cluster.annotation <- read.table(
  file = "../results/tables/cite_seq/cohort_treatment_naive/cluster_annotate/cluster_annotation_final.tsv",
  sep = "\t",
  header = TRUE,
  row.names = 1,
  stringsAsFactors = FALSE
) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      cluster_coarse,
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      cluster_main,
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

### Filtered dataset

In [ ]:
seu <- subset(seu, subset = cluster_main %in% c("B", "DC", "ILC", "Mono", "NK", "T"))
seu[[]] <- droplevels(seu[[]])

In [ ]:
# Group sample time points into conditions
seu$condition <- factor(
  case_match(
    seu$hash_id,
    "1" ~ "Baseline",
    "2" ~ "Early",
    "3" ~ "Late",
    "4" ~ "Late",
    "5" ~ "Late"
  ),
  levels = c("Baseline", "Early", "Late")
)

In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "26.1 Gb"

In [ ]:
seu

An object of class Seurat 
26712 features across 264512 samples within 3 assays 
Active assay: RNA (26424 features, 4993 variable features)
 3 layers present: data, counts, scale.data
 2 other assays present: ADT, ADTC
 5 dimensional reductions calculated: pca, integrated.rna, apca, integrated.adt, umap

In [ ]:
dir.create("../data/processed/cite_seq/cohort_treatment_naive/annotated", recursive = TRUE)

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/cite_seq/cohort_treatment_naive/annotated/annotated.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

In [ ]:
write.h5ad(
  seu,
  file = "../data/processed/cite_seq/cohort_treatment_naive/annotated/annotated_rna.h5ad",
  assay = "RNA",
  X = "counts",
  layers = NULL,
  obsm = list(X_pca = "integrated.rna")
)

Loading Python libraries

Extracting data from Seurat object

Creating AnnData object

Writing AnnData object to file



In [ ]:
write.h5ad(
  seu,
  file = "../data/processed/cite_seq/cohort_treatment_naive/annotated/annotated_adt.h5ad",
  assay = "ADT",
  X = "counts",
  layers = NULL,
  obsm = list(X_pca = "integrated.adt")
)

Loading Python libraries

Extracting data from Seurat object

Creating AnnData object

Writing AnnData object to file



# Treatment non-responder

### Full dataset

QS object generated by running Snakemake workflows in `01_prepare_internal_datasets`

In [ ]:
seu <- qs::qread(file = "../data/processed/cite_seq/cohort_nonresponders/cellbender/multi/peak-method:None_b:False_d:False/n-features:5000_normalisation:LogNormalize_clr:seurat_M:True_C:True/batch:orig.ident_normalisation:LogNormalize_integration:harmony/integrated.qs", nthreads = getOption("hpc.ncpus", 1))

### Cluster annotation

In [ ]:
cluster.annotation <- read.table(
  file = "../results/tables/cite_seq/cohort_nonresponders/cluster_annotate/cluster_annotation_final.tsv",
  sep = "\t",
  header = TRUE,
  row.names = 1,
  stringsAsFactors = FALSE
) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      cluster_coarse,
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      cluster_main,
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

### Filtered dataset

In [ ]:
seu <- subset(seu, subset = cluster_main %in% c("B", "DC", "ILC", "Mono", "NK", "T"))
seu[[]] <- droplevels(seu[[]])

In [ ]:
seu <- subset(seu, subset = donor == 3530194) # treatment-naive donor only

In [ ]:
# Group sample time points into conditions
seu$condition <- factor(
  case_match(
    seu$hash_id,
    "1" ~ "Baseline",
    "2" ~ "Early",
    "3" ~ "Late",
    "4" ~ "Late",
    "5" ~ "Late",
    "Relapse" ~ "Relapse"
  ),
  levels = c("Baseline", "Early", "Late", "Relapse")
)

In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "2.7 Gb"

In [ ]:
seu

An object of class Seurat 
26692 features across 45338 samples within 3 assays 
Active assay: RNA (26424 features, 4989 variable features)
 3 layers present: scale.data, data, counts
 2 other assays present: ADT, ADTC
 6 dimensional reductions calculated: pca, integrated.rna, apca, integrated.adt, ref.pca, umap

In [ ]:
dir.create("../data/processed/cite_seq/cohort_nonresponders/annotated", recursive = TRUE)

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/cite_seq/cohort_nonresponders/annotated/annotated.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

In [ ]:
write.h5ad(
  seu,
  file = "../data/processed/cite_seq/cohort_nonresponders/annotated/annotated_rna.h5ad",
  assay = "RNA",
  X = "counts",
  layers = NULL,
  obsm = list(X_pca = "integrated.rna")
)

# CSF/blood validation dataset

### Full dataset

QS object generated by running notebooks in `02_prepare_external_datasets`

In [ ]:
seu <- qs::qread(file = "../data/processed/external/cantoni/annotated/cantoni_untreated_ms_hc.qs", nthreads = getOption("hpc.ncpus", 1))

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/external/cantoni/annotated/cantoni_untreated_ms_hc.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

### Cluster annotation

In [ ]:
cluster.annotation <- read.table(
  file = "../results/tables/external/cantoni/cluster_annotate/cluster_annotation_final.tsv",
  sep = "\t",
  header = TRUE,
  row.names = 1,
  stringsAsFactors = FALSE
) %>%
  select(cluster_fine) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_macrophage", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        c("B_naive_transitional", "B_naive") ~ "B_naive",
        "B_memory" ~ "B_memory",
        "B_plasma" ~ "B_plasma",
        "DC_AXL_SIGLEC6" ~ "DC_AXL_SIGLEC6",
        c("DC_conventional_1", "DC_conventional_2") ~ "DC_conventional",
        "DC_plasmacytoid" ~ "DC_plasmacytoid",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_macrophage", "Mono_CD14_CD16") ~ "Mono_CD14",
        c("Mono_CD16", "Mono_CD16_IFN") ~ "Mono_CD16",
        "NK_CD56bright" ~ "NK_CD56bright",
        "NK_CD56dim" ~ "NK_CD56dim",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN") ~ "T_CD4_naive",
        c("T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector") ~ "T_CD4_memory",
        "T_regulatory_naive" ~ "T_regulatory_naive",
        "T_regulatory_memory" ~ "T_regulatory_memory",
        c("T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN") ~ "T_CD8_naive",
        c("T_CD8_memory_central", "T_CD8_memory_effector") ~ "T_CD8_memory",
        "T_MAIT" ~ "T_MAIT",
        "T_GD" ~ "T_GD",
        "T_DN" ~ "T_DN"
      ),
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        c("B_naive", "B_memory", "B_plasma") ~ "B",
        c("DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid") ~ "DC",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD16") ~ "Mono",
        c("NK_CD56bright", "NK_CD56dim") ~ "NK",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_memory", "T_regulatory_naive", "T_regulatory_memory",
          "T_CD8_naive", "T_CD8_memory", "T_MAIT", "T_GD", "T_DN") ~ "T"
      ),
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

### Filtered dataset

In [ ]:
seu <- subset(seu, subset = cluster_main %in% c("B", "DC", "ILC", "Mono", "NK", "T"))
seu[[]] <- droplevels(seu[[]])

In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "12.2 Gb"

In [ ]:
seu

An object of class Seurat 
18841 features across 178027 samples within 1 assay 
Active assay: RNA (18841 features, 0 variable features)
 3 layers present: counts, scale.data, data
 3 dimensional reductions calculated: pca, integrated.rna, umap

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/external/cantoni/annotated/cantoni_untreated_ms_hc_filtered.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

In [ ]:
write.h5ad(
  seu,
  file = "../data/processed/external/cantoni/annotated/cantoni_untreated_ms_hc_filtered.h5ad",
  assay = "RNA",
  X = "counts",
  layers = NULL,
  obsm = list(X_pca = "integrated.rna")
)

Loading Python libraries



Extracting data from Seurat object

Creating AnnData object

Writing AnnData object to file



# Brain tissue lymphocytes validation dataset

### Full dataset

QS object generated by running notebooks in `02_prepare_external_datasets`

In [ ]:
seu <- qs::qread(file = "../data/processed/external/lesion_rims/annotated/lesion_rims_lymphocytes.qs", nthreads = getOption("hpc.ncpus", 1))

In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "0.1 Gb"

### Cluster annotation

In [ ]:
cluster.annotation <- read.table(
  file = "../results/tables/external/lesion_rims/cluster_annotate/cluster_annotation_final.tsv",
  sep = "\t",
  header = TRUE,
  row.names = 1,
  stringsAsFactors = FALSE
) %>%
  select(cluster_fine) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        c("B_naive_transitional", "B_naive") ~ "B_naive",
        "B_memory" ~ "B_memory",
        "B_plasma" ~ "B_plasma",
        "DC_AXL_SIGLEC6" ~ "DC_AXL_SIGLEC6",
        c("DC_conventional_1", "DC_conventional_2") ~ "DC_conventional",
        "DC_plasmacytoid" ~ "DC_plasmacytoid",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16") ~ "Mono_CD14",
        c("Mono_CD16", "Mono_CD16_IFN") ~ "Mono_CD16",
        "NK_CD56bright" ~ "NK_CD56bright",
        "NK_CD56dim" ~ "NK_CD56dim",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN") ~ "T_CD4_naive",
        c("T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector") ~ "T_CD4_memory",
        "T_regulatory_naive" ~ "T_regulatory_naive",
        "T_regulatory_memory" ~ "T_regulatory_memory",
        c("T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN") ~ "T_CD8_naive",
        c("T_CD8_memory_central", "T_CD8_memory_effector") ~ "T_CD8_memory",
        "T_MAIT" ~ "T_MAIT",
        "T_GD" ~ "T_GD",
        "T_DN" ~ "T_DN"
      ),
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        c("B_naive", "B_memory", "B_plasma") ~ "B",
        c("DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid") ~ "DC",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD16") ~ "Mono",
        c("NK_CD56bright", "NK_CD56dim") ~ "NK",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_memory", "T_regulatory_naive", "T_regulatory_memory",
          "T_CD8_naive", "T_CD8_memory", "T_MAIT", "T_GD", "T_DN") ~ "T"
      ),
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

### Filtered dataset

In [ ]:
seu <- subset(seu, subset = cluster_main %in% c("B", "T"))
seu[[]] <- droplevels(seu[[]])

In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "0.1 Gb"

In [ ]:
seu

An object of class Seurat 
20521 features across 1417 samples within 1 assay 
Active assay: RNA (20521 features, 0 variable features)
 3 layers present: counts, scale.data, data
 3 dimensional reductions calculated: pca, integrated.rna, umap

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/external/lesion_rims/annotated/lesion_rims_lymphocytes_filtered.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

In [ ]:
write.h5ad(
  seu,
  file = "../data/processed/external/lesion_rims/annotated/lesion_rims_lymphocytes_filtered.h5ad",
  assay = "RNA",
  X = "counts",
  layers = NULL,
  obsm = list(X_pca = "integrated.rna")
)

Loading Python libraries

Extracting data from Seurat object



Creating AnnData object

Writing AnnData object to file



# Natalizumab PBMC validation dataset

### Full dataset

QS object generated by running notebooks in `02_prepare_external_datasets`

In [ ]:
seu <- qs::qread(file = "../data/processed/external/kaufmann/annotated/kaufmann_full.qs", nthreads = getOption("hpc.ncpus", 1))

### Cluster annotation

In [ ]:
cluster.annotation <- read.table(
  file = "../results/tables/external/kaufmann/cluster_annotate/cluster_annotation_final.tsv",
  sep = "\t",
  header = TRUE,
  row.names = 1,
  stringsAsFactors = FALSE
) %>%
  select(cluster_fine) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        c("B_naive_transitional", "B_naive") ~ "B_naive",
        "B_memory" ~ "B_memory",
        "B_plasma" ~ "B_plasma",
        "DC_AXL_SIGLEC6" ~ "DC_AXL_SIGLEC6",
        c("DC_conventional_1", "DC_conventional_2") ~ "DC_conventional",
        "DC_plasmacytoid" ~ "DC_plasmacytoid",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16") ~ "Mono_CD14",
        c("Mono_CD16", "Mono_CD16_IFN") ~ "Mono_CD16",
        "NK_CD56bright" ~ "NK_CD56bright",
        "NK_CD56dim" ~ "NK_CD56dim",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN") ~ "T_CD4_naive",
        c("T_CD4_memory_central", "T_CD4_memory_effector") ~ "T_CD4_memory",
        "T_regulatory_naive" ~ "T_regulatory_naive",
        "T_regulatory_memory" ~ "T_regulatory_memory",
        c("T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN") ~ "T_CD8_naive",
        c("T_CD8_memory_central", "T_CD8_memory_effector") ~ "T_CD8_memory",
        "T_MAIT" ~ "T_MAIT",
        "T_GD" ~ "T_GD",
        "T_DN" ~ "T_DN"
      ),
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        c("B_naive", "B_memory", "B_plasma") ~ "B",
        c("DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid") ~ "DC",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD16") ~ "Mono",
        c("NK_CD56bright", "NK_CD56dim") ~ "NK",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_memory", "T_regulatory_naive", "T_regulatory_memory",
          "T_CD8_naive", "T_CD8_memory", "T_MAIT", "T_GD", "T_DN") ~ "T"
      ),
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

### Filtered dataset

In [ ]:
seu <- subset(seu, subset = cluster_main %in% c("B", "DC", "ILC", "Mono", "NK", "T"))
seu[[]] <- droplevels(seu[[]])

In [ ]:
for (x in c("sample_date", "birth_year", "age_sampling", "sex", "EDSS", "MRI_n_CE_lesions", "disease_onset", "prev_treatments")) {
  seu[[x]] <- NULL
}
write.h5ad(
  seu,
  file = "../data/processed/external/kaufmann/annotated/kaufmann_full_filtered_rna.h5ad",
  assay = "RNA",
  X = "counts",
  layers = NULL,
  obsm = list(X_pca = "integrated.rna")
)

In [ ]:
seu <- subset(seu, subset = cohort == "NAT_HI" & group != "HI1" & donor != "HH-OX-31") # subset to MS NAT donors and remove donor without pre-treatment sample

In [ ]:
# Group sample time points into conditions
seu$condition <- factor(
  if_else(
    seu$natalizumab_treatment == "yes",
    "Post",
    "Pre"
  ),
  levels = c("Pre", "Post")
)

In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "4 Gb"

In [ ]:
seu

An object of class Seurat 
15212 features across 56252 samples within 2 assays 
Active assay: RNA (15174 features, 0 variable features)
 3 layers present: counts, scale.data, data
 1 other assay present: ADT
 5 dimensional reductions calculated: pca, integrated.rna, apca, integrated.adt, umap

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/external/kaufmann/annotated/kaufmann_natalizumab_filtered.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

# Session Info

In [5]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /ceph/project/fuggerlab/rfarooq/.conda/envs/sandbox/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
[1] C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] reticulate_1.39.0  lubridate_1.9.3    forcats_1.0.0      stringr_1.5.1     
 [5] dplyr_1.1.4        purrr_1.0.2        readr_2.1.5        tidyr_1.3.1       
 [9] tibble_3.2.1       ggplot2_3.5.1      tidyverse_2.0.0    future_1.34.0     
[13] Seurat_5.1.0       SeuratObject_5.0.2 sp_2.1-4          

loaded via a namespace (and not attached):
  [1] RColorBrewer_1.1-3     jsonlite_1.8.9         magrittr_2.0.3        
  [4] spatstat.utils_3.1-0   farver_2.1.2           vctrs_0.6.5           
  [7] ROCR_1.0-11            spatstat.explore_3.2-6 base6